# 02 · Demo 2 — one job → one verdict → one exit code

`./release-gate.sh` submits the release-gate job, reads back the per-benchmark scores, rolls them into **one weighted verdict**, syncs the record to MLflow, and **exits 1 (BLOCKED) or 0 (PROMOTE)** — a real CI gate.

We're on candidate **rc1**, so this should fail: a behavioral miss *and* a critical OWASP system-prompt-leak finding drag the weighted score below threshold.

In [ ]:
import os
from pathlib import Path

# Notebooks live in <repo>/notebooks; walk up to the dir that holds the Makefile.
root = Path.cwd()
while not (root / "Makefile").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working dir:", Path.cwd())

## Run the gate (offline, rc1)
We capture and print the exit code — that exit code is the entire point: it's what stops a release in CI.

In [ ]:
!./release-gate.sh --offline; ec=$?; echo; echo "gate exit code: $ec"

### Read the banner
- Weighted score **0.66 < threshold 0.75 → RELEASE BLOCKED**, **exit 1**.
- `owasp_llm_top10` **FAILs** — the system-prompt leak (we fix this in notebook 03).
- `agent_trace_judge` **FAILs** — the behavioral miss from Demo 1.
- The passing benchmarks (truthfulness, capability) show it's not *all* bad — which is exactly why a single rolled-up verdict matters.

In a pipeline this is literally `- run: ./release-gate.sh` — the release stops here, the same way a red unit test blocks a merge.

## Optional — inspect the raw verdict
The gate prints pure JSON to stdout (status lines go to stderr), so `jq` can slice it.

In [ ]:
!python3 -m harness.evalhub_client gate --mode offline 2>/dev/null | jq '.results.test'

Next: `03-harden-and-promote.ipynb` — apply the fix and watch the same gate flip green.